In [ ]:
!nvidia-smi

Mon Jun 22 07:20:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
%%writefile example.cu
// EXAMPLE 1: Hello World
// Printing in GPU
#include <stdio.h>

//kernel function which will runs in GPU
__global__ void example() {
  printf("BLOCK ID %d and THREAD ID %d\n",blockIdx.x, threadIdx.x );

}

int main() {
    printf("CUDA Program\n");
    // KERNEL_NAME<<<NUM_OF_BLOCKS,NUM_OF_THREADS>>> ();
    example <<<1,2>>> ();
    cudaDeviceSynchronize();
    return 0;
}

Writing example.cu


In [ ]:
!nvcc example.cu -o example
!./example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CUDA Program
BLOCK ID 0 and THREAD ID 0
BLOCK ID 0 and THREAD ID 1


In [ ]:
%%writefile example.cu
// EXAMPLE 2: T4 supports upto 1024 thread in each block. Let's see what happen if exceeds.
// Printing in GPU
#include <stdio.h>

//kernel function which will runs in GPU
__global__ void example() {
  printf("BLOCK ID %d and THREAD ID %d\n",blockIdx.x, threadIdx.x );

}

int main() {
    printf("CUDA Program\n");
    // KERNEL_NAME<<<NUM_OF_BLOCKS,NUM_OF_THREADS>>> ();
    example <<<1,1025>>> (); // Didn't print anything
    example <<<1,2>>> (); // Worked
    cudaDeviceSynchronize();
    return 0;
}

Overwriting example.cu


In [ ]:
!nvcc example.cu -o example
!./example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CUDA Program
BLOCK ID 0 and THREAD ID 0
BLOCK ID 0 and THREAD ID 1


In [ ]:
%%writefile example.cu
// EXAMPLE 3: In this example we will print blockIdx, threadIdx and wrapIdx
// Printing in GPU
#include <stdio.h>

//kernel function which will runs in GPU
__global__ void example() {
  int wrapIdx = 0;
  wrapIdx = threadIdx.x/32; // 32 blockIdx is default in every generations mostly.
  printf("BLOCK ID %d and THREAD ID %d and WRAP ID %d\n",blockIdx.x, threadIdx.x, wrapIdx);

}

int main() {
    printf("CUDA Program\n");
    // KERNEL_NAME<<<NUM_OF_BLOCKS,NUM_OF_THREADS>>> ();
    // example <<<1,128>>> (); // Total Wraps: 4
    // example <<<1,64>>> (); // Total Wraps: 2
    example <<<1,32>>> (); // Total Wraps: 1


    cudaDeviceSynchronize();
    return 0;
}

Overwriting example.cu


In [ ]:
!nvcc example.cu -o example
!./example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CUDA Program
BLOCK ID 0 and THREAD ID 0 and WRAP ID 0
BLOCK ID 0 and THREAD ID 1 and WRAP ID 0
BLOCK ID 0 and THREAD ID 2 and WRAP ID 0
BLOCK ID 0 and THREAD ID 3 and WRAP ID 0
BLOCK ID 0 and THREAD ID 4 and WRAP ID 0
BLOCK ID 0 and THREAD ID 5 and WRAP ID 0
BLOCK ID 0 and THREAD ID 6 and WRAP ID 0
BLOCK ID 0 and THREAD ID 7 and WRAP ID 0
BLOCK ID 0 and THREAD ID 8 and WRAP ID 0
BLOCK ID 0 and THREAD ID 9 and WRAP ID 0
BLOCK ID 0 and THREAD ID 10 and WRAP ID 0
BLOCK ID 0 and THREAD ID 11 and WRAP ID 0
BLOCK ID 0 and THREAD ID 12 and WRAP ID 0
BLOCK ID 0 and THREAD ID 13 and WRAP ID 0
BLOCK ID 0 and THREAD ID 14 and WRAP ID 0
BLOCK ID 0 and THREAD ID 15 and WRAP ID 0
BLOCK ID 0 and THREAD ID 16 and WRAP ID 0
BLOCK ID 0 and THREAD ID 17 and WRAP ID 0
BLOCK ID 0 and THREAD ID 18 and WRAP ID 0
BLOCK ID 0 a

In [11]:
%%writefile example.cu
// EXAMPLE 3: Performing Vector Additions
// Printing in GPU
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
// #define SIZE 1024
#define SIZE 32

//kernel function which will runs in GPU
__global__ void example(int *A, int *B, int *C, int n) {
  int i=threadIdx.x;
  C[i]=A[i]+B[i];
}

int main() {
    printf("CUDA Program\n");
    // KERNEL_NAME<<<NUM_OF_BLOCKS,NUM_OF_THREADS>>> ();
    int *d_a, *d_b, *d_c; // DEVICE VECTORS
    int size = SIZE*sizeof(int);

    // Dynamic Memory Allocation On CPU
    int *a = (int*)malloc(size);
    int *b = (int*)malloc(size);
    int *c = (int*)malloc(size);

    // Dynamic Memory Allocation On GPU
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);

    // Initialize The Inputs
    for (int i=0; i<SIZE; i++){
      a[i]=i;
      b[i]=SIZE-i;
    }

    // COPY VALUE FROM CPU TO GPU MEMORY
    cudaMemcpy(d_a,a,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_b,b,size,cudaMemcpyHostToDevice);

    example <<<1,1024>>> (d_a,d_b,d_c,SIZE);
    cudaDeviceSynchronize();

    cudaMemcpy(c,d_c,size,cudaMemcpyDeviceToHost);
    printf("EXECUTION FINISHED\n");
    for (int i=0;i<SIZE;i++){
      printf("%d + %d = %d", a[i],b[i],c[i]);
      printf("\n");
    }

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    free(a);
    free(b);
    free(c);
    return 0;
}

Overwriting example.cu


In [12]:
!nvcc example.cu -o example
!./example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CUDA Program
EXECUTION FINISHED
0 + 32 = 32
1 + 31 = 32
2 + 30 = 32
3 + 29 = 32
4 + 28 = 32
5 + 27 = 32
6 + 26 = 32
7 + 25 = 32
8 + 24 = 32
9 + 23 = 32
10 + 22 = 32
11 + 21 = 32
12 + 20 = 32
13 + 19 = 32
14 + 18 = 32
15 + 17 = 32
16 + 16 = 32
17 + 15 = 32
18 + 14 = 32
19 + 13 = 32
20 + 12 = 32
21 + 11 = 32
22 + 10 = 32
23 + 9 = 32
24 + 8 = 32
25 + 7 = 32
26 + 6 = 32
27 + 5 = 32
28 + 4 = 32
29 + 3 = 32
30 + 2 = 32
31 + 1 = 32


In [17]:
%%writefile example.cu
// EXAMPLE 4: Performing Vector Additions with around 2048 elements each.
// Printing in GPU
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
// #define SIZE 2048 // COMMENTED TO AVOID REAL OUTPUT
#define SIZE 32 // TO GET SMALL OUTPUT


//kernel function which will runs in GPU
__global__ void example(int *A, int *B, int *C, int n) {
  int i=threadIdx.x+(blockIdx.x*blockDim.x);
  C[i]=A[i]+B[i];
}

int main() {
    printf("CUDA Program\n");
    // KERNEL_NAME<<<NUM_OF_BLOCKS,NUM_OF_THREADS>>> ();
    int *d_a, *d_b, *d_c; // DEVICE VECTORS
    int size = SIZE*sizeof(int);

    // Dynamic Memory Allocation On CPU
    int *a = (int*)malloc(size);
    int *b = (int*)malloc(size);
    int *c = (int*)malloc(size);

    // Dynamic Memory Allocation On GPU
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);

    // Initialize The Inputs
    for (int i=0; i<SIZE; i++){
      a[i]=i;
      b[i]=SIZE-i;
    }

    // COPY VALUE FROM CPU TO GPU MEMORY
    cudaMemcpy(d_a,a,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_b,b,size,cudaMemcpyHostToDevice);

    example <<<2,32>>> (d_a,d_b,d_c,SIZE);
    cudaDeviceSynchronize();

    cudaMemcpy(c,d_c,size,cudaMemcpyDeviceToHost);
    printf("EXECUTION FINISHED\n");
    for (int i=0;i<SIZE;i++){
      printf("%d + %d = %d", a[i],b[i],c[i]);
      printf("\n");
    }

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    free(a);
    free(b);
    free(c);
    return 0;
}

Overwriting example.cu


In [18]:
!nvcc example.cu -o example
!./example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CUDA Program
EXECUTION FINISHED
0 + 32 = 32
1 + 31 = 32
2 + 30 = 32
3 + 29 = 32
4 + 28 = 32
5 + 27 = 32
6 + 26 = 32
7 + 25 = 32
8 + 24 = 32
9 + 23 = 32
10 + 22 = 32
11 + 21 = 32
12 + 20 = 32
13 + 19 = 32
14 + 18 = 32
15 + 17 = 32
16 + 16 = 32
17 + 15 = 32
18 + 14 = 32
19 + 13 = 32
20 + 12 = 32
21 + 11 = 32
22 + 10 = 32
23 + 9 = 32
24 + 8 = 32
25 + 7 = 32
26 + 6 = 32
27 + 5 = 32
28 + 4 = 32
29 + 3 = 32
30 + 2 = 32
31 + 1 = 32


In [23]:
%%writefile example.cu
// EXAMPLE 5: Measuring Execution Time In Vector Addition Using Events.
// Printing in GPU
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <cuda.h>
#include <device_launch_parameters.h>

// #define SIZE 2048 // COMMENTED TO AVOID REAL OUTPUT
#define SIZE 32 // TO GET SMALL OUTPUT


//kernel function which will runs in GPU
__global__ void example(int *A, int *B, int *C, int n) {
  int i=threadIdx.x+(blockIdx.x*blockDim.x);
  C[i]=A[i]+B[i];
}

int main() {
    printf("CUDA Program\n");
    // KERNEL_NAME<<<NUM_OF_BLOCKS,NUM_OF_THREADS>>> ();
    int *d_a, *d_b, *d_c; // DEVICE VECTORS
    int size = SIZE*sizeof(int);

    // Dynamic Memory Allocation On CPU
    int *a = (int*)malloc(size);
    int *b = (int*)malloc(size);
    int *c = (int*)malloc(size);

    // Dynamic Memory Allocation On GPU
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);

    // Initialize The Inputs
    for (int i=0; i<SIZE; i++){
      a[i]=i;
      b[i]=SIZE-i;
    }

    // COPY VALUE FROM CPU TO GPU MEMORY
    cudaMemcpy(d_a,a,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_b,b,size,cudaMemcpyHostToDevice);

    // CREATE EVENT
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    example <<<2,32>>> (d_a,d_b,d_c,SIZE);
    cudaEventRecord(stop);
    cudaDeviceSynchronize();

    float millisec = 0;
    cudaEventElapsedTime(&millisec,start,stop);
    printf("Execution Time In Millisec %f\n", millisec);



    cudaMemcpy(c,d_c,size,cudaMemcpyDeviceToHost);
    printf("EXECUTION FINISHED\n");
    for (int i=0;i<SIZE;i++){
      printf("%d + %d = %d", a[i],b[i],c[i]);
      printf("\n");
    }

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    free(a);
    free(b);
    free(c);
    return 0;
}

Overwriting example.cu


In [24]:
!nvcc example.cu -o example
!./example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CUDA Program
Execution Time In Millisec 0.189856
EXECUTION FINISHED
0 + 32 = 32
1 + 31 = 32
2 + 30 = 32
3 + 29 = 32
4 + 28 = 32
5 + 27 = 32
6 + 26 = 32
7 + 25 = 32
8 + 24 = 32
9 + 23 = 32
10 + 22 = 32
11 + 21 = 32
12 + 20 = 32
13 + 19 = 32
14 + 18 = 32
15 + 17 = 32
16 + 16 = 32
17 + 15 = 32
18 + 14 = 32
19 + 13 = 32
20 + 12 = 32
21 + 11 = 32
22 + 10 = 32
23 + 9 = 32
24 + 8 = 32
25 + 7 = 32
26 + 6 = 32
27 + 5 = 32
28 + 4 = 32
29 + 3 = 32
30 + 2 = 32
31 + 1 = 32


In [44]:
%%writefile example.cu
// EXAMPLE 6: Measuring Execution Time In Large Vector Addition Using Events.
// Printing in GPU
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <cuda.h>
#include <device_launch_parameters.h>
#define SIZE 1024*432*1024


//kernel function which will runs in GPU
__global__ void example(int *A, int *B, int *C, int n) {
  int i=threadIdx.x+(blockIdx.x*blockDim.x);
  C[i]=A[i]+B[i];
}

int main() {
    printf("CUDA Program\n");
    // KERNEL_NAME<<<NUM_OF_BLOCKS,NUM_OF_THREADS>>> ();
    int *d_a, *d_b, *d_c; // DEVICE VECTORS
    int size = SIZE*sizeof(int);
    printf("Stage 1\n");

    // Dynamic Memory Allocation On GPU
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);
    printf("Stage 2\n");

    // Dynamic Memory Allocation On CPU
    int *a = (int*)malloc(size);
    int *b = (int*)malloc(size);
    int *c = (int*)malloc(size);
    printf("Stage 3\n");

    // Initialize The Inputs
    for (int i=0; i<SIZE; i++){
      a[i]=i;
      b[i]=SIZE-i;
    }

    // COPY VALUE FROM CPU TO GPU MEMORY
    cudaMemcpy(d_a,a,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_b,b,size,cudaMemcpyHostToDevice);

    // CREATE EVENT
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    example <<<1024*432,1024>>> (d_a,d_b,d_c,SIZE);
    cudaEventRecord(stop);
    cudaDeviceSynchronize();

    float millisec = 0;
    cudaEventElapsedTime(&millisec,start,stop);
    printf("Execution Time In Millisec %f\n", millisec);



    cudaMemcpy(c,d_c,size,cudaMemcpyDeviceToHost);
    // printf("EXECUTION FINISHED\n");
    // for (int i=0;i<SIZE;i++){
    //  printf("%d + %d = %d", a[i],b[i],c[i]);
    //  printf("\n");
    // }

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    free(a);
    free(b);
    free(c);
    return 0;
}

Overwriting example.cu


In [45]:
!nvcc example.cu -o example
!./example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CUDA Program
Stage 1
Stage 2
Stage 3
Execution Time In Millisec 21.299360


In [46]:
!ncu ./example

CUDA Program
Stage 1
==PROF== Connected to process 29831 (/content/example)
Stage 2
Stage 3
==PROF== Profiling "example" - 0: 0%....50%....100% - 9 passes
Execution Time In Millisec 2803.885254
==PROF== Disconnected from process 29831
[29831] example@127.0.0.1
  example(int *, int *, int *, int) (442368, 1, 1)x(1024, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- -------------
    Metric Name             Metric Unit  Metric Value
    ----------------------- ----------- -------------
    DRAM Frequency                  Ghz          5.00
    SM Frequency                    Mhz        585.00
    Elapsed Cycles                cycle    12,395,305
    Memory Throughput                 %         89.25
    DRAM Throughput                   %         89.25
    Duration                         ms         21.19
    L1/TEX Cache Throughput           %         29.92
    L2 Cache Throughput               %         29.31

In [34]:
%%writefile example.cu
// EXAMPLE 6: Measuring Execution Time In Large Vector Addition Using Events.
// NOTE: IT WILL FAIL. T4 DOESN'T HAVE THAT MUCH MEMORY.
// Printing in GPU
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <cuda.h>
#include <device_launch_parameters.h>
// #define SIZE 1024*432*1024 // TO GET SMALL OUTPUT
#define SIZE 1024*1024*1024 // TO GET SMALL OUTPUT


//kernel function which will runs in GPU
__global__ void example(int *A, int *B, int *C, int n) {
  int i=threadIdx.x+(blockIdx.x*blockDim.x);
  C[i]=A[i]+B[i];
}

int main() {
    printf("CUDA Program\n");
    // KERNEL_NAME<<<NUM_OF_BLOCKS,NUM_OF_THREADS>>> ();
    int *d_a, *d_b, *d_c; // DEVICE VECTORS
    int size = SIZE*sizeof(int);
    printf("Stage 1\n");

    // Dynamic Memory Allocation On GPU
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);
    printf("Stage 2\n");

    // Dynamic Memory Allocation On CPU
    int *a = (int*)malloc(size);
    int *b = (int*)malloc(size);
    int *c = (int*)malloc(size);
    printf("Stage 3\n");

    // Initialize The Inputs
    for (int i=0; i<SIZE; i++){
      a[i]=i;
      b[i]=SIZE-i;
    }

    // COPY VALUE FROM CPU TO GPU MEMORY
    cudaMemcpy(d_a,a,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_b,b,size,cudaMemcpyHostToDevice);

    // CREATE EVENT
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    example <<<1024*432,1024>>> (d_a,d_b,d_c,SIZE);
    cudaEventRecord(stop);
    cudaDeviceSynchronize();

    float millisec = 0;
    cudaEventElapsedTime(&millisec,start,stop);
    printf("Execution Time In Millisec %f\n", millisec);



    cudaMemcpy(c,d_c,size,cudaMemcpyDeviceToHost);
    // printf("EXECUTION FINISHED\n");
    // for (int i=0;i<SIZE;i++){
    //  printf("%d + %d = %d", a[i],b[i],c[i]);
    //  printf("\n");
    // }

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    free(a);
    free(b);
    free(c);
    return 0;
}

Overwriting example.cu


In [35]:
!nvcc example.cu -o example
!./example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
example.cu(23): warning #69-D: integer conversion resulted in truncation
      int size = 1024*1024*1024*sizeof(int);
                 ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

example.cu(23): warning #69-D: integer conversion resulted in truncation
      int size = 1024*1024*1024*sizeof(int);
                 ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

example.cu: In function ‘int main()’:
example.cu:23:36: warning: overflow in conversion from ‘long unsigned int’ to ‘int’ changes value from ‘4294967296’ to ‘0’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Woverflow-Woverflow]8;;]
   23 |     int size = SIZE*sizeof(int);
      |            ~~~~~~~~~~~~~~~~~~~~~   ^             
CUDA Program
Stage 1
St

In [ ]:
%%writefile example.cu
// PENDING
// EXAMPLE 7: Measuring Execution Time In Large Vector Addition Using Events.
// NOTE: TO HANDLE VERY LARGE DATASET WHAT WE CAN DO IS TO DIVIDE IT IN CHUNKS BECAUSE RAM & VRAM IS NOT ENOUGH.
// Printing in GPU
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <cuda.h>
#include <device_launch_parameters.h>
#define SIZE 1024*1024*1024
#define CHUNK_SIZE 1024*1024*128


//kernel function which will runs in GPU
__global__ void example(int *A, int *B, int *C, int n) {
  int i=threadIdx.x+(blockIdx.x*blockDim.x);
  C[i]=A[i]+B[i];
}

int main() {
    printf("CUDA Program\n");
    // KERNEL_NAME<<<NUM_OF_BLOCKS,NUM_OF_THREADS>>> ();
    int *d_a, *d_b, *d_c; // DEVICE VECTORS
    int size = SIZE*sizeof(int);
    printf("Stage 1\n");

    // Dynamic Memory Allocation On GPU
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);
    printf("Stage 2\n");

    // Dynamic Memory Allocation On CPU
    int *a = (int*)malloc(size);
    int *b = (int*)malloc(size);
    int *c = (int*)malloc(size);
    printf("Stage 3\n");

    // Initialize The Inputs
    for (int i=0; i<SIZE; i++){
      a[i]=i;
      b[i]=SIZE-i;
    }

    // COPY VALUE FROM CPU TO GPU MEMORY
    cudaMemcpy(d_a,a,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_b,b,size,cudaMemcpyHostToDevice);

    // CREATE EVENT
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    example <<<1024*432,1024>>> (d_a,d_b,d_c,SIZE);
    cudaEventRecord(stop);
    cudaDeviceSynchronize();

    float millisec = 0;
    cudaEventElapsedTime(&millisec,start,stop);
    printf("Execution Time In Millisec %f\n", millisec);



    cudaMemcpy(c,d_c,size,cudaMemcpyDeviceToHost);
    // printf("EXECUTION FINISHED\n");
    // for (int i=0;i<SIZE;i++){
    //  printf("%d + %d = %d", a[i],b[i],c[i]);
    //  printf("\n");
    // }

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    free(a);
    free(b);
    free(c);
    return 0;
}

In [ ]:
!nvcc example.cu -o example
!./example

In [36]:
%%writefile example.cu
// EXAMPLE 8: CUDA RUNTIME API: GET TOTAL GPUs Count
#include <stdio.h>
#include <cuda_runtime.h>

int main() {
    int deviceCount = 0;

    cudaError_t err = cudaGetDeviceCount(&deviceCount);

    if (err != cudaSuccess) {
        printf("CUDA Error: %s\n",
               cudaGetErrorString(err));
        return -1;
    }

    printf("Number of CUDA Devices: %d\n",
           deviceCount);

    return 0;
}

Overwriting example.cu


In [37]:
!nvcc example.cu -o example
!./example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Number of CUDA Devices: 1


In [40]:
%%writefile example.cu
// EXAMPLE 9: CUDA RUNTIME API: Print Device Name
#include <stdio.h>
#include <cuda_runtime.h>

int main() {

    int nDevices;

    cudaGetDeviceCount(&nDevices);

    printf("Total GPUs = %d\n\n", nDevices);

    for (int i = 0; i < nDevices; i++) {

        cudaDeviceProp prop;

        cudaGetDeviceProperties(&prop, i);

        printf("Device %d\n", i);
        printf("Name: %s\n", prop.name);
        printf("Compute Capability: %d.%d\n", prop.major, prop.minor);
        printf("Global Memory: %.2f GB\n", (float)prop.totalGlobalMem /(1024*1024*1024));
        printf("SM Count: %d\n", prop.multiProcessorCount);
        printf("Max Threads/Block: %d\n",prop.maxThreadsPerBlock);
        printf("\n");
    }

    return 0;
}

Overwriting example.cu


In [41]:
!nvcc example.cu -o example
!./example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Total GPUs = 1

Device 0
Name: Tesla T4
Compute Capability: 7.5
Global Memory: 14.56 GB
SM Count: 40
Max Threads/Block: 1024

